# Nowcast EMAE: Probar Fetchers

In [ ]:
import pandas as pd
import requests
import yfinance as yf
from io import StringIO

In [11]:
def fetch_cammesa(start='2020-01-01'):
    """
    Descarga la demanda real diaria del SADI (MW) desde datos.gob.ar
    y la convierte a consumo diario en GWh (promedio MW × 24 h / 1000).
    """
    import pandas as pd
    import requests

    # 1. Consultar la API CKAN para obtener los recursos del dataset
    api = (
        "https://datos.gob.ar/api/3/action/package_show"
        "?id=energia-datos-compania-administradora-mercado-mayorista-electrico-sa-cammesa"
    )
    res = requests.get(api, timeout=30).json()['result']['resources']

    # 2. Encontrar el recurso HTML con la tabla de "Demanda Real y Prevista del SADI"
    html_url = next(
        r['url']
        for r in res
        if 'Demanda Real y Prevista' in r.get('name','') and r['format'].lower() == 'html'
    )

    # 3. Leer la tabla con pandas
    tables = pd.read_html(html_url, parse_dates=[0], dayfirst=True)
    df = tables[0]  # primera tabla: columnas ['Fecha', 'Real', 'Prevista']
    df.columns = [c.lower().strip() for c in df.columns]
    df = df[['fecha', 'real']].rename(columns={'real': 'value'})
    df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True)
    df = df.set_index('fecha').sort_index()

    # 4. Filtrar por fecha de inicio y agregar a diario en GWh
    df = df.loc[df.index >= pd.to_datetime(start)]
    df_daily = (df['value'].resample('D').mean() * 24 / 1000).to_frame(name='gwh')

    return df_daily


In [13]:
# Probar fetch_cammesa
cam = fetch_cammesa()
cam.tail()

StopIteration: 

In [4]:
def fetch_sube():
    api = ("https://datos.transporte.gob.ar/api/3/action/package_show"
           "?id=sube-cantidad-de-transacciones-usos-por-fecha")
    resources = requests.get(api, timeout=30).json()['result']['resources']
    csv_urls = [r['url'] for r in resources if r['format'].lower() == 'csv']
    frames = []
    for url in csv_urls:
        df = pd.read_csv(url)
        df['fecha'] = pd.to_datetime(df['fecha'])
        frames.append(df[['fecha', 'cantidad_transacciones']])
    sube = (pd.concat(frames)
              .groupby('fecha')['cantidad_transacciones']
              .sum().rename('viajes_sube')
              .to_frame()
              .sort_index())
    return sube

In [5]:
# Probar fetch_sube
sube = fetch_sube()
sube.tail()

KeyError: 'fecha'

In [6]:
def fetch_bcra(series="base_monetaria"):
    token = "TU_TOKEN_BCRA"  # reemplaza con tu token
    url = f"https://api.estadisticasbcra.com/{series}"
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=30)
    r.raise_for_status()
    s = pd.read_json(StringIO(r.text))
    s['d'] = pd.to_datetime(s['d'])
    return s.set_index('d').rename(columns={'v': series})

In [ ]:
# Probar fetch_bcra (base_monetaria)
bm = fetch_bcra()
bm.tail()

In [7]:
def fetch_ccl(start='2020-01-01'):
    adr = yf.download('GGAL', start=start, progress=False)['Adj Close'] # type: ignore
    loc = yf.download('GGAL.BA', start=start, progress=False)['Adj Close'] # type: ignore
    return (adr * 10 / loc).rename('ccl_ggal').to_frame()

In [8]:
# Probar fetch_ccl
ccl = fetch_ccl()
ccl.tail()

YF.download() has changed argument auto_adjust default to True



1 Failed download:
['GGAL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


KeyError: 'Adj Close'

In [9]:
def fetch_soybeans():
    url = "https://stooq.com/q/d/l/?s=zs=F&i=d"
    soy = pd.read_csv(url)
    soy['Date'] = pd.to_datetime(soy['Date'])
    return soy.set_index('Date')['Close'].rename('soy_usd').to_frame()

In [10]:
# Probar fetch_soybeans
soy = fetch_soybeans()
soy.tail()

KeyError: 'Date'

In [ ]:
# Construir panel completo y revisar
panel = pd.concat([
    fetch_cammesa(), fetch_sube(), fetch_bcra(),
    fetch_ccl(), fetch_soybeans()
], axis=1)
panel.tail()